In [ ]:
# es/data-analysis/normal/07-groupby-basics
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("titanic.csv", ())


## El patrón dividir-aplicar-combinar

GroupBy es una de las características más potentes de pandas. Sigue un patrón de tres pasos:

1. **Dividir** — divide el DataFrame en grupos según una o más columnas
2. **Aplicar** — calcula una función en cada grupo de forma independiente
3. **Combinar** — une los resultados de nuevo en un solo DataFrame


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")


## Agrupando por una sola columna


In [ ]:
# Average survival rate by passenger class
print(df.groupby("Pclass")["Survived"].mean())


Salida:


In [ ]:
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64


Los pasajeros de primera clase tuvieron una tasa de supervivencia del 63%, en comparación con el 24% de la tercera clase. Groupby reveló una diferencia de clase marcada en segundos.

**Qué sucede paso a paso:**


In [ ]:
# This is conceptually what groupby does:
for pclass, group_df in df.groupby("Pclass"):
    print(f"Class {pclass}: {group_df['Survived'].mean():.3f}")


## Agrupando por varias columnas


In [ ]:
# Survival rate by class and sex
print(df.groupby(["Pclass", "Sex"])["Survived"].mean())


Salida:


In [ ]:
Pclass  Sex   
1       female    0.968085
        male      0.368852
2       female    0.921053
        male      0.157407
3       female    0.500000
        male      0.135447
Name: Survived, dtype: float64


Usa `unstack()` para hacer esto más fácil de leer:


In [ ]:
print(df.groupby(["Pclass", "Sex"])["Survived"].mean().unstack())


## Métodos de agregación

Groupby admite todas las agregaciones estándar:


In [ ]:
# Mean fare by class
print(df.groupby("Pclass")["Fare"].mean())

# Total fare collected per class
print(df.groupby("Pclass")["Fare"].sum())

# Count of passengers per class
print(df.groupby("Pclass")["PassengerId"].count())


## Múltiples agregaciones con agg()

El método `agg()` aplica varias funciones a la vez:


In [ ]:
print(df.groupby("Pclass")["Fare"].agg(["mean", "median", "min", "max", "count"]))


Salida:


In [ ]:
              mean  median     min       max  count
Pclass                                             
1        84.154687  60.287  0.0000  512.3292    216
2        20.662183  19.575  0.0000   73.5000    184
3        13.675550   8.050  0.0000   56.4958    491


**Diferentes agregaciones por columna:**


In [ ]:
print(df.groupby("Pclass").agg({
    "Survived": "mean",
    "Fare": ["mean", "max"],
    "Age": "median",
    "Name": "count"
}))


## Agregando todas las columnas numéricas


In [ ]:
# Quick summary of all numeric columns per group
print(df.groupby("Pclass").mean(numeric_only=True))


## GroupBy con filtros

Después de agrupar, puedes filtrar grupos completos:


In [ ]:
# Keep only groups with more than 50 passengers
large_groups = df.groupby("Pclass").filter(lambda x: len(x) > 50)
print(large_groups["Pclass"].value_counts())


## Inténtalo

Usando el conjunto de datos del Titanic, calcula:
1. La tarifa promedio para cada puerto de embarque
2. La tasa de supervivencia para cada combinación de sexo y puerto de embarque
3. Las estadísticas de edad (media, mediana, mín, máx) para cada clase de pasajero


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")

print("Average fare by port:")
print(df.groupby("Embarked")["Fare"].mean())

print("\nSurvival by sex and port:")
print(df.groupby(["Sex", "Embarked"])["Survived"].mean().unstack())

print("\nAge stats by class:")
print(df.groupby("Pclass")["Age"].agg(["mean", "median", "min", "max"]))


## Conclusiones clave

- GroupBy sigue el patrón dividir-aplicar-combinar: divide los datos, aplica una función, combina los resultados
- Agrupa por una columna para resúmenes simples y por varias columnas para análisis más profundos
- `agg()` te permite calcular varias estadísticas a la vez, por columna si es necesario
- Groupby revela patrones invisibles en los datos crudos

## Desafío de práctica

Del conjunto de datos del Titanic, calcula la tasa de supervivencia para cada combinación de Pclass, Sex y si el pasajero viajaba solo (SibSp + Parch == 0). ¿Qué grupo tuvo la mayor tasa de supervivencia? ¿Cuál tuvo la menor?


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
